# Boston Housing Analysis

**Tabular Regression Project** — Analyze and predict Boston housing prices.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Boston Housing (506 rows × 14 features)
Target: `medv` (median value in $1000s)

## 2 · Project Overview

This project provides a **deeper analytical perspective** on the Boston Housing dataset. While the sibling 'Classification' project focuses on model comparison, this notebook emphasizes **EDA**, **feature relationships**, and **model interpretation** alongside the standard regression pipeline.

We explore multicollinearity, feature distributions, and spatial patterns in the data.

## 3 · Learning Objectives

1. Conduct thorough exploratory data analysis on housing data.
2. Investigate multicollinearity and its impact on linear models.
3. Compare ridge and lasso regularization with unregularized regression.
4. Build gradient-boosting models and interpret feature importance.
5. Use LazyPredict and FLAML for automated benchmarking.

## 4 · Problem Statement

Analyze the relationships between Boston suburb characteristics and housing prices, then build a predictive model for **median house value** (`medv`).

## 5 · Why This Project Matters

- Understanding **what drives housing prices** is essential for policy, lending, and urban planning.
- Multicollinearity analysis teaches an important concept often skipped in quick ML projects.
- Comparing regularized vs. unregularized linear models builds regression intuition.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 506 |
| **Features** | 13 numeric features (crim, zn, indus, chas, nox, rm, age, dis, rad, tax, ptratio, black, lstat) |
| **Target** | medv (median value in $1000s) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: UCI ML Repository, originally Harrison & Rubinfeld (1978).
- **License**: Public domain.
- **Ethical note**: The `black` variable is ethically problematic in modern contexts.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'medv'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
# Load from sibling folder
DATA_PATH = os.path.join(SAVE_DIR, 'Boston Dataset.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(SAVE_DIR, '..', 'Boston Housing Prediction Analysis', 'Boston Dataset.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min()}, {df[TARGET].max()}]')
print(f'\nFeature stats:')
print(df.describe().T[['mean','std','min','max']])

## 13 · Exploratory Data Analysis

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation_heatmap.png'), dpi=100)
plt.show()

# Key scatter plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, ['rm', 'lstat', 'ptratio', 'crim', 'nox', 'dis']):
    ax.scatter(df[col], df[TARGET], alpha=0.4, s=10)
    ax.set_xlabel(col); ax.set_ylabel(TARGET)
    ax.set_title(f'{col} vs {TARGET}')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_scatter.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].hist(bins=30, ax=axes[0], edgecolor='black')
axes[0].set_title(f'{TARGET} Distribution')
axes[0].axvline(50, color='r', linestyle='--', label='Cap at 50')
axes[0].legend()

import scipy.stats as stats
stats.probplot(df[TARGET], plot=axes[1])
axes[1].set_title('Q-Q Plot')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'target_analysis.png'), dpi=100)
plt.show()
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split Strategy

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 16 · Preprocessing Strategy

All features are numeric. No encoding needed. We note that rad/tax are highly correlated (multicollinear).

In [ ]:
# Check multicollinearity
high_corr = df.corr().abs()
high_pairs = []
for i in range(len(high_corr.columns)):
    for j in range(i+1, len(high_corr.columns)):
        if high_corr.iloc[i,j] > 0.7:
            high_pairs.append((high_corr.columns[i], high_corr.columns[j], high_corr.iloc[i,j]))
print('Highly correlated pairs (>0.7):')
for a, b, c in high_pairs:
    print(f'  {a} — {b}: {c:.3f}')

## 17 · Feature Engineering

In [ ]:
for df_part in [X_train, X_test]:
    df_part['rm_sq'] = df_part['rm'] ** 2
    df_part['lstat_log'] = np.log1p(df_part['lstat'])

print(f'Features after engineering: {X_train.shape[1]}')

## 18 · Baseline Model

We compare OLS, Ridge, and Lasso as baselines.

In [ ]:
for name, model in [('OLS', LinearRegression()), ('Ridge', Ridge(alpha=1.0)), ('Lasso', Lasso(alpha=0.1))]:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f'{name}: RMSE={root_mean_squared_error(y_test, preds):.2f}, '
          f'MAE={mean_absolute_error(y_test, preds):.2f}, '
          f'R²={r2_score(y_test, preds):.4f}')

baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **rm** (rooms) has the strongest positive relationship with price.
- **lstat** (% lower-status) has the strongest negative relationship.
- Multicollinearity between rad and tax doesn't affect tree models but inflates linear model coefficients.
- Policy implication: investing in neighborhoods' socioeconomic conditions could raise property values.

## 26 · Limitations

- 1970s data — outdated for modern Boston.
- medv censored at 50.
- Ethically problematic variables.
- Only 506 observations.

## 27 · How to Improve This Project

1. Use VIF analysis to formally quantify multicollinearity.
2. Try target transformation (log of medv).
3. Cross-validate all models.
4. Spatial analysis using dis/rad features.

## 28 · Production Considerations

- Would need current Census data.
- Fair lending regulations constrain feature use.
- Spatial models (kriging) might be more appropriate.
- Need confidence intervals for valuations.

## 29 · Common Mistakes

1. Not checking for censoring at medv=50.
2. Ignoring multicollinearity in linear models.
3. Not scaling features for Ridge/Lasso.
4. Treating the `chas` binary variable as continuous.

## 30 · Mini Challenge / Exercises

1. Compute VIF for all features and discuss.
2. Compare log-transformed vs. raw medv predictions.
3. Build a model without the `black` variable — does accuracy change?
4. Plot partial dependence plots for the top 3 features.

## 31 · Final Summary / Key Takeaways

- Thorough EDA reveals multicollinearity and censoring issues.
- Gradient-boosting models handle these issues naturally.
- rm and lstat are the dominant predictors.
- Regularization (Ridge/Lasso) helps linear models on correlated features.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['Baseline_LR'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')